# 175 — IA para programación y modernización

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — pass@k

```text
n=6, c=3 → C(3,k)/C(6,k)
pass@1 = 1 − 3/6            = 0.50
pass@2 = 1 − C(3,2)/C(6,2)  = 1 − 3/15  = 0.80
pass@4 = 1 − C(3,4)/C(6,4)  = 1 − 0/15  = 1.00   (C(3,4)=0: imposible elegir 4 malas de 3)

Ingenuo: k=2 → 1 − 0.5² = 0.75 ;  k=4 → 1 − 0.5⁴ = 0.9375
```

c) El ingenuo **subestima**: trata las muestras como independientes con reemplazo,
pero elegir k de un conjunto fijo sin reemplazo agota las malas más rápido. Con
k=4 la fórmula exacta ya garantiza 1.0 (solo hay 3 incorrectas) y el ingenuo no.


In [ ]:
from math import comb
n, c = 6, 3
pass_at = lambda k: 1 - comb(n - c, k) / comb(n, k)
for k in (1, 2, 4):
    print(k, pass_at(k), 1 - (1 - c / n) ** k)
assert pass_at(4) == 1.0


## Solución 2 — Veredicto del harness

a) **Rechazado.** La regla es conjuntiva: TODOS los FAIL_TO_PASS deben pasar Y TODOS
los PASS_TO_PASS deben seguir pasando. `test_cache` falla → el parche introduce una
regresión.

b) Modificar tests previos cambia la definición del contrato en lugar de cumplirlo;
el harness fija la suite del repo original justamente para impedirlo. Un revisor
humano trataría ese diff como bandera roja máxima: el agente está optimizando la
métrica, no el software.

c) Ejemplo: el issue pide "parsear números negativos"; el parche añade
`if s == "-1": return -1` (hardcode del caso del test). Pasa FAIL_TO_PASS, no toca
nada más (PASS_TO_PASS intactos) y es incorrecto para cualquier otro negativo. Los
tests son una muestra del contrato, no el contrato.


In [ ]:
fail_to_pass = {"test_parse_neg": True, "test_parse_zero": True}
pass_to_pass = {"test_render": True, "test_cache": False}
aceptado = all(fail_to_pass.values()) and all(pass_to_pass.values())
print("aceptado:", aceptado)
assert aceptado is False


## Solución 3 — Muestrear y filtrar

a) El papel de los tests lo juega el **contrato JSON verificable**: claves
obligatorias, `evidence` inspeccionable, `limitations` presente. Igual que en
pass@k, un candidato solo cuenta si un verificador programático lo acepta; la
semilla documenta la reproducibilidad de cada muestra.

b) Filtro de referencia abajo — nota que verifica estructura, no valores internos:
esa es la diferencia entre contrato y sobreajuste al caso.


In [ ]:
def filtro(resultado):
    return (
        resultado.get("kind") == "frontier"
        and bool(resultado.get("evidence"))
        and bool(resultado.get("limitations"))
    )

for seed in (175, 275, 375):
    r = run_lab("frontier", seed=seed)
    assert filtro(r), seed
print("3/3 candidatos pasan el contrato")


## Solución 4 — Orden de modernización

1. **Congelar referencia ejecutable** — sin un original reproducible no hay contra
   qué comparar nada.
2. **Generar tests de caracterización** — fijan el comportamiento real (bugs
   incluidos); son la definición ejecutable de "equivalente".
3. **Traducir módulo a módulo con LLM** — unidades pequeñas, cada una validada
   contra los tests del paso 2.
4. **Comparar salidas nuevo vs referencia** — equivalencia conductual de extremo a
   extremo con los mismos insumos.
5. **Revisión manual de dinero y zonas horarias** — ⚠️ este paso NUNCA se delega
   sin revisión humana: el redondeo decimal (float vs decimal) y las zonas horarias
   son donde la traducción "plausible" produce diferencias silenciosas y caras.
